In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_02_errors_and_autocorrelation.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 02: Ergodicity, Statistical vs. Systematic Errors, Autocorrelation, Data Blocking & Resampling

_Session 5dbe8e69 · Estimated time: ~120 minutes · 42 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Define ergodicity in deterministic and stochastic simulations, relate the equilibrium distribution F*(S) to the canonical Boltzmann distribution, and distinguish statistical errors from systematic errors.
- Calculate the normalized autocorrelation function C(t), integrated autocorrelation time τ_int, and statistical inefficiency κ = 1 + 2∑ C(t) ≈ 1 + 2τ for correlated MD/MC time series.
- Derive the variance of the sample mean for correlated data, compute the effective number of independent samples N_eff = N_eq / κ, and evaluate true standard errors σ_true = σ_naive √κ.
- Apply the Flyvbjerg–Petersen data blocking (binning) method to extract plateau standard errors, determine κ from (σ_plateau / σ_naive)², and diagnose why N_eff ≥ 20 is required.
- Select the equilibration warm-up cutoff k_1 (I_eq) to balance transient systematic drift against statistical variance, avoiding the smooth cumulative-average trap.
- Derive the O(N⁻¹) systematic bias and O(N⁻¹/²) statistical error of nonlinear functions F(Z̄) via Taylor expansion, and apply Jackknife and Bootstrap resampling to nonlinear observables.


In [2]:
ws.render_section('sec_ergodicity_and_error_types')

## 1. Ergodicity, Equilibrium Distributions & Statistical vs. Systematic Errors

In Lesson 01 we saw that any atomistic simulation — whether **deterministic** (integrating Newton's equations via Molecular Dynamics) or **stochastic** (sampling Markov transitions $T(S_{n+1}\mid S_n)$ via Monte Carlo or Langevin dynamics) — generates a sequence of states along iteration index $n$ ("time" or "Monte Carlo time"):
$$S_0 \Longrightarrow S_1 \Longrightarrow S_2 \Longrightarrow \cdots \Longrightarrow S_n \Longrightarrow S_{n+1} \Longrightarrow \cdots$$
Regardless of whether the underlying rule is deterministic or stochastic, **we analyze the errors of the output trace the exact same way**.

### The Ergodic Hypothesis & Equilibrium Distribution $F^*(S)$
Why can a single long trajectory starting from one arbitrary initial configuration $S_0$ tell us the macroscopic thermodynamic properties of a material? Because we assume the simulation is **ergodic** (Lecture 02, Slides 2–3):
- Let $F_t(S \mid S_0)$ be the probability distribution of finding the system in state $S$ after time $t$, given initial state $S_0$.
- A simulation is **ergodic** if, after a characteristic **correlation time** $\kappa$ (the number of iterations required to "forget" its past), the system loses all memory of its initial state $S_0$ (except for exact conserved quantities like total energy $E$ and total momentum $\mathbf{P}$ in an isolated $NVE$ run) and converges to a **unique equilibrium probability distribution** $F^*(S)$:
  $$\lim_{t \to \infty} F_t(S \mid S_0) = F^*(S)$$
- In classical statistical mechanics at constant temperature $T$, this unique distribution is the **canonical Boltzmann distribution**:
  $$F^*(S) = Z^{-1} \exp\!\left(-\frac{\mathcal{H}(S)}{k_B T}\right), \qquad Z = \int dS\, \exp\!\left(-\frac{\mathcal{H}(S)}{k_B T}\right)$$
- **Two uses of ergodicity in every simulation:**
  1. **Equilibration (Warm-Up):** Starting from an artificial $S_0$ (e.g., a perfect lattice when simulating a liquid), waiting for $t_{\text{eq}} \gg \kappa$ allows the system to relax into $F^*(S)$.
  2. **Independent Sampling:** Once in equilibrium, any non-conserved observable $A(S_t)$ sampled at time separations $\Delta t \gg \kappa$ behaves as if drawn independently from $F^*(S)$, allowing us to replace the phase-space integral $\langle V(S)\rangle_{F^*} = \int dS\, F^*(S)V(S)$ with a time average!
- Besides static thermodynamic averages (like internal energy $U = \langle V(S)\rangle_{F^*}$), trajectories also yield dynamical transport coefficients such as the **diffusion constant** $D$ in $d$ spatial dimensions:
  $$D = \lim_{t \to \infty} \frac{1}{2 d t} \big\langle |\mathbf{r}(t) - \mathbf{r}(0)|^2 \big\rangle$$

### Statistical Errors vs. Systematic Errors
In what sense do simulations calculate "exact" properties? If there are no systematic biases, averaging long enough ($N \to \infty$) drives the random sampling error to zero! However, *"simulation results without error bars are only suggestive — without error bars one has no idea of their significance."* Every serious simulation must distinguish two fundamentally different sources of error (Slide 14):

| Error Category | Physical / Mathematical Origin | Scaling with Run Length $N$ | Atomistic Simulation Examples |
| :--- | :--- | :--- | :--- |
| **Statistical Error** | Finite random sampling of thermal fluctuations around $\langle A\rangle$ | Decays as $\sigma_{\bar{A}} = \sqrt{\frac{\kappa_2 \kappa}{N}} \propto N^{-1/2}$ | Finite number of MD/MC steps; variance of block averages |
| **Systematic Error** | Non-sampling bias in the model, algorithm, or analysis; **does NOT vanish as $N \to \infty$** (except $O(1/N)$ nonlinear estimator bias) | Independent of $N$ (persists even if you sample forever!) | Empirical potential inaccuracy; MD timestep $\Delta t$ truncation; finite box size ($1/N_{\text{atoms}}$); un-discarded warm-up transient; floating-point round-off; code bugs |

### Computational Efficiency $\zeta$ of a Simulation Method
How do we compare two different algorithms (e.g., two Monte Carlo move sets) that compute the same quantity? Since statistical error shrinks as $\sigma_{\text{err}}(T) \propto 1/\sqrt{T}$ where $T$ is the total CPU wall-clock time spent, the product $T \cdot \sigma_{\text{err}}^2$ is **independent of the length of the run**. We define the **computational efficiency** $\zeta$ (Slide 14):
$$\zeta = \frac{1}{T \, \sigma_{\text{err}}^2}$$
Efficiency $\zeta$ depends on the computer hardware, the algorithm, the correlation time $\kappa$, and the observable being measured, **but NOT on how long you run the simulation**!


In [3]:
ws.render_section('q_1')

**Q1 · easy** · 💡

What does it mean for an atomistic simulation $S_0 \to S_1 \to \cdots \to S_t$ to be **ergodic** (Lecture 02, Slides 2–3)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how the conditional state distribution $F_t(S \mid S_0)$ behaves for times $t \gg \kappa$.

</div>
</details>

- [ ] **(A)** For times $t \gg \kappa$ (much longer than the correlation time), the system loses memory of its initial state $S_0$ (aside from conserved quantities) and samples a unique equilibrium probability distribution $\lim_{t\to\infty} F_t(S\mid S_0) = F^*(S)$.
- [ ] **(B)** The instantaneous potential energy $V(S_t)$ becomes strictly constant and stops fluctuating once $t > \kappa$.
- [ ] **(C)** Every microstate in phase space has the exact same energy $\mathcal{H}(S)$ regardless of temperature.
- [ ] **(D)** Consecutive states $S_n$ and $S_{n+1}$ are completely uncorrelated at a single step separation ($\Delta n = 1$).

In [4]:
ws.render_section('q_2')

**Q2 · easy** · 💡

According to Lecture 02 (Slide 2), how does proving ergodicity compare between **stochastic simulations** (like Markov Chain Monte Carlo) and **deterministic simulations** (like Newtonian Molecular Dynamics)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think of the famous 1954 Fermi–Pasta–Ulam anharmonic chain experiment in deterministic mechanics vs. Markov chain transition matrices.

</div>
</details>

- [ ] **(A)** Ergodicity is often straightforward to prove mathematically for random/stochastic transitions, but is notoriously difficult (or fails in quasi-integrable regimes) for deterministic simulations.
- [ ] **(B)** Ergodicity is trivial to prove for every deterministic Newtonian system, including harmonic crystals, but impossible to prove for Monte Carlo.
- [ ] **(C)** Deterministic simulations can never be ergodic under any circumstances.
- [ ] **(D)** Ergodicity only applies to quantum systems, not classical phase space.

In [5]:
ws.render_section('q_3')

**Q3 · medium** · 💡

In a 3D ($d = 3$) Molecular Dynamics simulation of liquid argon, you measure the mean-squared displacement of atoms at long times ($t = 200\text{ ps}$) to be $\langle |\mathbf{r}(t) - \mathbf{r}(0)|^2 \rangle = 180.0\text{ Å}^2$. Using the Einstein diffusion relation $D = \lim_{t\to\infty} \frac{1}{2dt}\langle |\mathbf{r}(t) - \mathbf{r}(0)|^2 \rangle$ (Lecture 02, Slide 3), what is the self-diffusion constant $D$ (in $\text{Å}^2/\text{ps}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In $d = 3$ dimensions, the denominator is $2 d t = 6 t$. Compute $180.0 / (6 \times 200)$.

</div>
</details>

_Numeric answer required._

In [6]:
ws.render_section('q_4')

**Q4 · medium** · 💡

Which of the following are examples of **systematic errors** (rather than statistical sampling errors) in an atomistic simulation? Select **all** that apply.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Ask yourself: if I run the exact same simulation code with the exact same parameters for $N \to \infty$ steps, does this error vanish or persist?

</div>
</details>

- [ ] **(A)** Using an empirical pair potential that omits three-body covalent bonding interactions present in the real material.
- [ ] **(B)** Choosing a Verlet integration time step $\Delta t = 10\text{ fs}$ that is too large, introducing $\mathcal{O}(\Delta t^2)$ Hamiltonian truncation drift.
- [ ] **(C)** Including the initial non-equilibrium warm-up transient ($t < t_{\text{eq}}$) inside your production average.
- [ ] **(D)** The random $\pm \sigma_A / \sqrt{N_{\text{eff}}}$ uncertainty in the sample mean due to averaging over a finite number of equilibrium fluctuations.

In [7]:
ws.render_section('q_5')

**Q5 · medium** · 💡

In Lecture 02 (Slide 14), the **efficiency** of a simulation algorithm is defined as $\zeta = \frac{1}{T \sigma^2}$, where $T$ is the CPU time (in hours) and $\sigma$ is the achieved statistical error bar on the mean. Suppose Algorithm A runs for $T_A = 4.0\text{ hours}$ and achieves a statistical error bar of $\sigma_A = 0.05\text{ eV}$, while Algorithm B runs for $T_B = 2.0\text{ hours}$ and achieves $\sigma_B = 0.025\text{ eV}$. What is the efficiency ratio $\zeta_B / \zeta_A$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\zeta_A = 1 / (4.0 \times 0.05^2)$ and $\zeta_B = 1 / (2.0 \times 0.025^2)$, then divide $\zeta_B / \zeta_A$.

</div>
</details>

_Numeric answer required._

In [8]:
ws.render_section('q_6')

**Q6 · easy** · 💡

If you take the exact same simulation code on the same computer and simply run it **4 times longer** ($T \to 4T$), how does the algorithmic efficiency $\zeta = \frac{1}{T \sigma_{\text{err}}^2}$ change?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

When $T \to 4T$, how does the statistical error bar $\sigma_{\text{err}} \propto 1/\sqrt{T}$ scale?

</div>
</details>

- [ ] **(A)** $\zeta$ remains unchanged, because $\sigma_{\text{err}}$ decreases by a factor of $2$ ($\sigma_{\text{err}}^2 \to \sigma_{\text{err}}^2 / 4$), canceling the $4\times$ increase in $T$.
- [ ] **(B)** $\zeta$ increases by a factor of $4$.
- [ ] **(C)** $\zeta$ decreases by a factor of $4$.
- [ ] **(D)** $\zeta$ increases by a factor of $2$.

In [9]:
ws.render_section('sec_autocorrelation_and_kappa')

### Why Naive Error Formulas Fail in MD and MC
Suppose we have an equilibrated trace of $N$ measurements $\{a_1, a_2, \dots, a_N\}$ with sample mean $\bar{a} = \frac{1}{N}\sum_{t=1}^N a_t$ and sample variance $\sigma^2(a_t) = \frac{1}{N-1}\sum_{t=1}^N (a_t - \bar{a})^2$. If the $N$ samples were **statistically independent**, the standard error of the mean would be the naive formula:
$$\text{error}_{\text{naive}}(\langle a \rangle) = \sqrt{\frac{\sigma^2(a_t)}{N}} = \left[\frac{\sum_{t=1}^N \delta a_t^2}{N(N-1)}\right]^{1/2}, \qquad \delta a_t \equiv a_t - \bar{a}$$
**Why is this dangerously wrong for raw MD or MC time series?** Consider the thought experiment on Slide 10 of Lecture 02: *imagine taking $N$ genuine samples and simply repeating (duplicating) every single measurement twice to create a dataset of $2N$ numbers!*
- The sample mean $\bar{a}$ is unchanged (still an unbiased estimator!).
- The sample variance $\sigma^2(a_t)$ is unchanged.
- Yet the naive formula $\sqrt{\sigma^2(a_t)/(2N)}$ claims your error bar just shrank by $\sqrt{2}$, even though you added **zero new information**!

In both Molecular Dynamics (where atoms move only $\sim 0.01\text{ Å}$ per femtosecond step) and Metropolis Monte Carlo (where only one particle or spin moves per step), consecutive configurations $S_t$ and $S_{t+1}$ are strongly correlated.

### Normalized Autocorrelation Function $C(t)$ & Correlation Time $\kappa$
Let $\delta a_t = a_t - \langle a \rangle$ denote the fluctuation of observable $a$ at step $t$. In equilibrium (time-translation invariance), the correlation between step $t$ and step $t'$ depends only on the time lag $\Delta t = |t - t'|$. We define the **normalized auto time-correlation function** $C(t)$ (Slides 11 & 16):
$$C(t, t') \equiv \frac{\langle \delta a_t \, \delta a_{t'} \rangle}{\langle \delta a_t^2 \rangle} = C(|t - t'|) = \frac{\big\langle (A_k - \langle A \rangle)(A_{k+t} - \langle A \rangle) \big\rangle}{\langle A^2 \rangle - \langle A \rangle^2}$$
For a finite equilibrated trace $A_k$ ($k_1 \le k \le k_2$, containing $N_{\text{eq}} = k_2 - k_1 + 1$ points) with sample mean $a$ and sample variance $v = \frac{1}{N_{\text{eq}}-1}\sum_{k=k_1}^{k_2} (A_k - a)^2$, the unbiased finite-sample estimator $c_A(i)$ at lag $i$ (Slide 16) is:
$$c_A(i) = \frac{1}{v} \frac{1}{N_{\text{eq}} - i} \sum_{k=k_1}^{k_2 - i} (A_k - a)(A_{k+i} - a)$$
Key properties of $C(t)$:
1. **At zero lag ($t = 0$):** $C(0) = \frac{\langle \delta a_t^2 \rangle}{\langle \delta a_t^2 \rangle} = 1$ identically.
2. **At large lag ($t \to \infty$):** In an ergodic system, $a_t$ and $a_{t+\Delta t}$ become statistically independent for $\Delta t \gg \tau$, so $\langle \delta a_t \delta a_{t+\Delta t}\rangle \to \langle \delta a_t\rangle \langle \delta a_{t+\Delta t}\rangle = 0 \implies C(t) \to 0$.
3. **Typical exponential decay:** Often $C(t) \approx \exp(-t / \tau)$, where $\tau$ is the **exponential decay correlation time** (in steps).

### Deriving the Correlated Error Formula & Statistical Inefficiency $\kappa$
Let's compute the true variance of the sample mean $\bar{a} = \frac{1}{N}\sum_{t=1}^N a_t$ WITHOUT assuming independence (Slide 11):
$$\big\langle (\bar{a} - \langle a \rangle)^2 \big\rangle = \left\langle \left(\frac{1}{N}\sum_{t=1}^N \delta a_t\right)\left(\frac{1}{N}\sum_{t'=1}^N \delta a_{t'}\right) \right\rangle = \frac{1}{N^2}\sum_{t=1}^N \sum_{t'=1}^N \langle \delta a_t \delta a_{t'} \rangle = \frac{\langle \delta a_t^2 \rangle}{N^2}\sum_{t=1}^N \sum_{t'=1}^N C(|t - t'|)$$
For a run much longer than the correlation decay ($N \gg \tau$), summing over the $N \times N$ square $(t, t')$ is bounded by summing $t$ from $1$ to $N$ and the relative lag $\Delta t = t' - t$ from $-\infty$ to $+\infty$:
$$\big\langle (\bar{a} - \langle a \rangle)^2 \big\rangle \approx \frac{\langle \delta a_t^2 \rangle}{N^2} \cdot N \sum_{\Delta t = -\infty}^{\infty} C(\Delta t) = \langle \delta a_t^2 \rangle \frac{\kappa}{N}$$
where we have defined the **statistical inefficiency** (called the **correlation time** $\kappa$ in Prof. Trinkle's slides):
$$\boxed{\kappa = \int_{-\infty}^{\infty} dt\, C(t) \approx \sum_{\Delta t = -\infty}^{\infty} C(\Delta t) = 1 + 2\sum_{t=1}^{\infty} C(t) = 1 + 2\tau_{\text{int}}}$$
Here $\tau_{\text{int}} = \sum_{t=1}^{\infty} C(t)$ is the **integrated autocorrelation time**. Notice how $\kappa$ relates to $\tau$:
- If data are **completely uncorrelated** ($C(t) = 0$ for all $t \ge 1$), then $\tau_{\text{int}} = 0$ and **$\kappa = 1$**.
- If $C(t) = e^{-t/\tau}$ with $\tau \gg 1$, then $\tau_{\text{int}} = \sum_{t=1}^\infty e^{-t/\tau} = \frac{1}{e^{1/\tau}-1} \approx \tau$, giving:
  $$\kappa \approx 1 + 2\tau$$
- Therefore, an equilibrated trace of length $N_{\text{eq}}$ contains only **$N_{\text{eff}}$ effective independent samples**:
  $$\boxed{N_{\text{eff}} = \frac{N_{\text{eq}}}{\kappa}, \qquad \sigma_{\bar{A}} = \sqrt{\frac{v_A}{N_{\text{eff}}}} = \sqrt{\frac{v_A \, \kappa}{N_{\text{eq}}}} = \sigma_{\text{naive}}\sqrt{\kappa}}$$

> [!WARNING]
> **Practical Cutoff Rule for $\kappa$ (Slides 11 & 16):**
> In a finite dataset, $c_A(i)$ at large lag $i$ fluctuates noisily around zero. Summing $c_A(i)$ all the way to $i = N_{\text{eq}}-1$ accumulates pure statistical noise (in fact, $\sum_{i=1}^{N-1} (1 - i/N)c_A(i) = -1/2$ identically!). Therefore, we **truncate the sum at a finite cutoff** $i_{\text{cutoff}}$ defined by the first lag where $c_A(i) \le 0$:
> $$\kappa = 1 + 2\sum_{i=1}^{i_{\text{cutoff}}} c_A(i)$$


In [10]:
ws.render_section('concept_autocorrelation')

### Interactive Explorer: Autocorrelation Decay C(t), Statistical Inefficiency κ & Effective Samples

Adjust the exponential correlation decay time $\tau$ and total steps $N$, and toggle between the **Autocorrelation Function $C(\Delta t)$** and the **Correlated Time Series Trace** (compare Lecture 02, Slide 12: *correlated data vs. uncorrelated data*).

**Key Takeaway:** When $\tau = 0$ (uncorrelated data, Slide 12 right), $C(\Delta t)$ drops immediately from $1$ at $\Delta t=0$ to $0$ at $\Delta t=1$, so $\kappa = 1$ and $N_{\text{eff}} = N$. When $\tau = 20$ steps, $\kappa \approx 1 + 2(20) = 41$, shrinking $N_{\text{eff}}$ by $41\times$ and inflating the true error bar by $\sqrt{41} \approx 6.4\times$ relative to the naive formula!

In [11]:
ws.render_section('q_7')

**Q7 · easy** · 💡

By definition of the normalized autocorrelation function $C(t) = \frac{\langle \delta A_k \delta A_{k+t}\rangle}{\langle \delta A_k^2 \rangle}$, what is the exact value of $C(0)$ at zero time lag ($t = 0$) for any non-constant observable $A$, and what value does $C(t)$ approach as $t \to \infty$ in an ergodic system?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $t = 0$ into the numerator and denominator of $C(t)$.

</div>
</details>

- [ ] **(A)** $C(0) = 1$ and $\lim_{t\to\infty} C(t) = 0$
- [ ] **(B)** $C(0) = \sigma_A^2$ and $\lim_{t\to\infty} C(t) = 1$
- [ ] **(C)** $C(0) = 0$ and $\lim_{t\to\infty} C(t) = \kappa$
- [ ] **(D)** $C(0) = \langle A \rangle^2$ and $\lim_{t\to\infty} C(t) = 0$

In [12]:
ws.render_section('q_8')

**Q8 · easy** · 💡

Consider the thought experiment from Lecture 02 (Slide 10): you have $N = 500$ genuinely independent measurements of energy, and you artificially repeat each number $4$ times in a row to create a trace of $N_{\text{total}} = 2{,}000$ correlated data points. What is the statistical inefficiency (correlation time) $\kappa$ of this $2{,}000$-point dataset?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall $N_{\text{eff}} = N_{\text{total}} / \kappa$. How many truly independent measurements are present in the dataset?

</div>
</details>

_Numeric answer required._

In [13]:
ws.render_section('q_9')

**Q9 · medium** · 💡

In a Monte Carlo simulation of a polymer chain, you compute the empirical autocorrelation function $c_A(i)$ of the end-to-end distance and find that it crosses zero after lag $i_{\text{cutoff}} = 4$, with values $c_A(1) = 0.70$, $c_A(2) = 0.45$, $c_A(3) = 0.25$, and $c_A(4) = 0.10$ (and $c_A(5) \le 0$). Using the Statistical Vocabulary table formula (Lecture 02, Slide 16):
$$\kappa = 1 + 2\sum_{i=1}^{i_{\text{cutoff}}} c_A(i)$$
what is the correlation time (statistical inefficiency) $\kappa$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Sum $c_A(1) + c_A(2) + c_A(3) + c_A(4) = 0.70 + 0.45 + 0.25 + 0.10$, multiply by $2$, and add $1$.

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('q_10')

**Q10 · medium** · 💡

An equilibrated Molecular Dynamics trajectory contains $N_{\text{eq}} = 10{,}000$ saved steps. The autocorrelation function of the pressure decays exponentially as $C(t) \approx e^{-t/\tau}$ with decay time $\tau = 12.0\text{ steps}$. Using the continuum/large-$\tau$ approximation $\kappa \approx 1 + 2\tau$, what is the **effective number of independent samples** $N_{\text{eff}} = N_{\text{eq}} / \kappa$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First compute $\kappa = 1 + 2(12.0) = 25.0$, then divide $N_{\text{eq}} = 10{,}000$ by $\kappa$.

</div>
</details>

_Numeric answer required._

In [15]:
ws.render_section('q_11')

**Q11 · medium** · 💡

In an MD simulation with $N_{\text{eq}} = 4{,}900$ equilibrated steps, the sample standard deviation of the total potential energy is $\sigma_A = \sqrt{v_A} = 1.40\text{ eV}$, and the correlation time (statistical inefficiency) is $\kappa = 49.0$. What is the **true standard error of the mean** $\sigma_{\bar{A}} = \sqrt{\frac{v_A \kappa}{N_{\text{eq}}}}$ (in $\text{eV}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $N_{\text{eff}} = N_{\text{eq}} / \kappa = 4900 / 49 = 100$, then divide $\sigma_A = 1.40\text{ eV}$ by $\sqrt{N_{\text{eff}}}$.

</div>
</details>

_Numeric answer required._

In [16]:
ws.render_section('q_12')

**Q12 · easy** · 💡

If a correlated simulation time series has a statistical inefficiency of $\kappa = 81$, by what multiplicative factor ($\sigma_{\text{true}} / \sigma_{\text{naive}}$) does the true standard error of the mean exceed the naive uncorrelated error estimate $s_A / \sqrt{N}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since $\sigma_{\text{true}} = \sigma_{\text{naive}} \sqrt{\kappa}$, take the square root of $\kappa = 81$.

</div>
</details>

_Numeric answer required._

In [17]:
ws.render_section('q_13')

**Q13 · medium** · 💡

Why must the summation $\kappa = 1 + 2\sum_{i=1}^{i_{\text{cutoff}}} c_A(i)$ be **truncated at a finite lag** $i_{\text{cutoff}}$ (where $c_A(i)$ first reaches zero) rather than summed all the way to the maximum lag $i = N_{\text{eq}} - 1$ (Lecture 02, Slides 11 & 16)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

At large lags $i \gg \tau$, the true $C(i)$ is zero, but each finite-sample estimate $c_A(i)$ has random statistical noise of order $1/\sqrt{N_{\text{eq}} - i}$.

</div>
</details>

- [ ] **(A)** At lags $i \gg \tau$, $c_A(i)$ consists of random finite-sample noise whose variance accumulates over hundreds of lags, corrupting the integral and biasing $\kappa$ toward zero if summed to $N_{\text{eq}}-1$.
- [ ] **(B)** Because $c_A(i)$ always diverges to $+\infty$ as $i \to N_{\text{eq}} - 1$.
- [ ] **(C)** Because $c_A(i)$ is only defined for $i = 1$ and $i = 2$.
- [ ] **(D)** Truncating at $C(t) = 0$ is done solely to save CPU time, as the large-lag terms are identically zero to machine precision.

In [18]:
ws.render_section('q_14')

**Q14 · medium** · 💡

Compare the two analysis windows shown on Slide 12 of Lecture 02: **correlated data (`data.2`)** on the left vs. **uncorrelated data (`data.1`)** on the right, both having $N = 1{,}000$ points. Which statement accurately describes the two datasets?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the `sigma`, `autocor. time`, and `error of mean` boxes on Slide 12 of Lecture 02.

</div>
</details>

- [ ] **(A)** The correlated dataset (`data.2`) has a much smaller intrinsic standard deviation (`sigma` $\approx 0.103$) than `data.1` (`sigma` $\approx 0.401$), yet because its autocorrelation time is $\kappa \approx 11.8$ (vs. $\kappa \approx 1.01$ for `data.1`), both datasets end up with nearly the same error of the mean ($\approx \pm 0.011$ vs. $\pm 0.013$)!
- [ ] **(B)** Both datasets have autocorrelation time $\kappa = 1.0$, so their error bars depend only on their means.
- [ ] **(C)** The uncorrelated dataset (`data.1`) has an autocorrelation function that decays slowly over 40 steps.
- [ ] **(D)** Because `data.2` has a smaller `sigma`, its true error of the mean is $10\times$ smaller than `data.1`.

In [19]:
ws.render_section('q_15')

**Q15 · hard** · 💡

For an exact discrete autoregressive AR(1) process where $C(t) = \phi^{|t|}$ with $0 < \phi < 1$ (corresponding to $\phi = e^{-1/\tau}$), the infinite series for the statistical inefficiency can be summed in closed form:
$$\kappa = 1 + 2\sum_{t=1}^{\infty} \phi^t = 1 + \frac{2\phi}{1 - \phi} = \frac{1 + \phi}{1 - \phi}$$
If neighboring steps have a one-step correlation of $\phi = C(1) = 0.80$, what is the exact statistical inefficiency $\kappa$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $(1 + 0.80) / (1 - 0.80)$.

</div>
</details>

_Numeric answer required._

In [20]:
ws.render_section('sec_data_blocking')

## 3. The Flyvbjerg–Petersen Data Blocking (Binning) Method & The N_eff ≥ 20 Rule

Computing the autocorrelation function $c_A(i)$ and choosing the cutoff lag $i_{\text{cutoff}}$ by hand is one way to find the true error bar. A second, remarkably elegant and automatic method that avoids computing $c_A(i)$ altogether is **Data Blocking (Binning)**, formalized by Flyvbjerg and Petersen (1989) (Lecture 02, Slide 10).

### How Data Blocking Works
Suppose we have an equilibrated trace of $N$ correlated measurements $\{a_1, a_2, \dots, a_N\}$. We partition the trace into $M = N / b$ contiguous, non-overlapping **blocks** each of length $b$ steps (note: Slide 10 denotes the block length by $N_b$, so there are $N / N_b$ blocks; in modern literature $b$ is the block length and $N_b = N/b$ is the number of blocks — both notations give the exact same formula!).

1. **Compute the average within each block $m \in \{1, 2, \dots, M\}$:**
   $$B_m = \frac{1}{b}\sum_{t=(m-1)b + 1}^{m b} a_t$$
   Notice that the overall mean of the block averages is identical to the original sample mean: $\bar{B} = \frac{1}{M}\sum_{m=1}^M B_m = \bar{a}$.
2. **Compute the sample variance among the $M$ block averages:**
   $$\sigma^2(B) = \frac{1}{M - 1}\sum_{m=1}^M (B_m - \bar{B})^2$$
3. **Estimate the standard error of the overall mean treating the $M = N/b$ blocks as independent:**
   $$\boxed{\hat{\sigma}_{\bar{a}}(b) = \sqrt{\frac{\sigma^2(B)}{M}} = \sqrt{\frac{b \, \sigma^2(B)}{N}}}$$

### Why Does the Blocking Curve Plateau at the True Error?
Let's trace what happens as we systematically increase the block length $b = 1, 2, 4, 8, 16, \dots$:
- **Small blocks ($b = 1$):** Each block is a single step ($M = N$), so $\hat{\sigma}_{\bar{a}}(1) = \sqrt{\sigma^2(a_t)/N} = \sigma_{\text{naive}}$. Because adjacent blocks are strongly correlated, this **underestimates** the true error by a factor of $\sqrt{\kappa}$.
- **Intermediate blocks ($1 < b \lesssim \tau$):** As $b$ grows, correlations *inside* each block are absorbed into the block variance $\sigma^2(B)$, causing $\hat{\sigma}_{\bar{a}}(b)$ to **climb monotonically upward**.
- **Plateau regime ($b \gg \tau$, typically $b \gtrsim 3\tau$ to $5\tau$):** Once each block is much longer than the correlation time $\tau$, adjacent block averages $B_m$ and $B_{m+1}$ become **statistically independent** (only a negligible boundary layer of width $\sim \tau$ at the interface between two blocks is correlated). Therefore, $\sigma^2(B) \to \frac{\sigma_A^2 \kappa}{b}$, and the blocking error estimate **converges to a flat plateau** equal to the true standard error:
  $$\hat{\sigma}_{\bar{a}}(b) \xrightarrow{b \gg \tau} \sigma_{\text{plateau}} = \sqrt{\frac{\sigma_A^2 \kappa}{N}} = \sigma_{\text{naive}}\sqrt{\kappa}$$
- **Extracting $\kappa$ directly from the blocking curve:** Once the plateau is reached, the ratio of the plateau error to the naive ($b=1$) error immediately yields the statistical inefficiency $\kappa$:
  $$\boxed{\kappa = \left(\frac{\sigma_{\text{plateau}}}{\sigma_{\text{naive}}}\right)^2}$$
- **Over-blocking Breakdown ($M = N/b < 20$):** If we make the block size $b$ so huge that only $M = 2, 3,$ or $5$ blocks remain in the entire simulation, the variance of the variance explodes! As stated on Slide 15 of Lecture 02:
  > *"We need about 20 independent data points to estimate errors (so error of error is only 20%)."*

  Specifically, for $M$ independent Gaussian blocks, the relative statistical uncertainty in the estimated standard error ("the error of the error") is:
  $$\frac{\Delta \hat{\sigma}_{\bar{a}}}{\hat{\sigma}_{\bar{a}}} \approx \frac{1}{\sqrt{2(M - 1)}}$$
  At $M = 20$ independent points (or $N_{\text{eff}} = 20$), $1/\sqrt{38} \approx 16.2\% \approx 20\%$. If $N_{\text{eff}} < 20$, no blocking plateau can form before the curve dissolves into wild finite-sample noise!


In [21]:
ws.render_section('concept_blocking')

### Interactive Explorer: Flyvbjerg–Petersen Data Blocking Curve & Plateau Extraction

Vary the correlation time $\tau$, total trajectory length $N$, and selected block length $b$ to watch $\hat{\sigma}_{\bar{A}}(b)$ climb from the naive $b=1$ underestimate to the true plateau $\sigma_A\sqrt{\kappa/N}$ — and observe what happens in the shaded noisy zone where fewer than $20$ blocks remain!

**Key Takeaway:** A reliable data-blocking error analysis requires two simultaneous conditions at your chosen plateau block size $b$: (1) $b \gg \tau$ so blocks are independent and the curve has flattened out, and (2) $N_b = N/b \ge 20$ so there are enough blocks to compute a trustworthy variance!

In [22]:
ws.render_section('q_16')

**Q16 · medium** · 💡

You partition an equilibrated MD trajectory of $N = 8{,}000$ steps into $M = 40$ contiguous blocks, each of length $b = N_b = 200$ steps (which is much longer than the correlation time $\tau = 15$ steps). You compute the block average $B_m$ for each of the $40$ blocks and find that the sample standard deviation among the 40 block averages is $\sigma(B) = 0.31623\text{ GPa}$ (so $\sigma^2(B) = 0.100\text{ GPa}^2$). Using the binning error formula (Lecture 02, Slide 10):
$$\text{error}(\langle a \rangle) \approx \sqrt{\frac{b\,\sigma^2(B)}{N}} = \sqrt{\frac{\sigma^2(B)}{M}}$$
what is the estimated standard error of the overall mean (in $\text{GPa}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Divide $\sigma^2(B) = 0.100$ by the number of blocks $M = 40$, then take the square root.

</div>
</details>

_Numeric answer required._

In [23]:
ws.render_section('q_17')

**Q17 · medium** · 💡

In a Flyvbjerg–Petersen data blocking analysis of an MC magnetization trace, the naive ($b=1$) standard error is $\hat{\sigma}_{\bar{M}}(1) = 0.0040$, and as block length $b$ increases, the blocking curve levels off at a plateau value of $\sigma_{\text{plateau}} = 0.0200$. What is the statistical inefficiency (correlation time) $\kappa$ of the magnetization trace?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use $\kappa = (\sigma_{\text{plateau}} / \sigma_{\text{naive}})^2$.

</div>
</details>

_Numeric answer required._

In [24]:
ws.render_section('q_18')

**Q18 · easy** · 💡

As you group an equilibrated time series into larger and larger blocks of size $b$, what happens to the **overall mean of the block averages** $\bar{B} = \frac{1}{M}\sum_{m=1}^M B_m$ (assuming $N$ is an exact multiple of $b$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the identity $\bar{b} = \bar{a}$ on Slide 10 of Lecture 02.

</div>
</details>

- [ ] **(A)** The mean of the block averages $\bar{B}$ is algebraically identical to the unblocked sample mean $\bar{a}$ for every block size $b$.
- [ ] **(B)** The mean of the block averages $\bar{B}$ increases as $\sqrt{b}$ until it reaches a plateau.
- [ ] **(C)** The mean of the block averages $\bar{B}$ shrinks as $1/b$.
- [ ] **(D)** Data blocking changes the sample mean by subtracting the correlation time $\kappa$.

In [25]:
ws.render_section('q_19')

**Q19 · medium** · 💡

Suppose a student runs an MD simulation for $N = 500$ steps on a slow glassy system whose correlation time is $\kappa = 200$ steps. When the student plots the data-blocking standard error $\hat{\sigma}_{\bar{A}}(b)$ versus block size $b$, why does the curve **never show a clear flat plateau**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How many effective independent samples $N_{\text{eff}} = N / \kappa$ are in the entire trajectory? Recall the rule on Slide 15 of Lecture 02.

</div>
</details>

- [ ] **(A)** Because $N_{\text{eff}} = N / \kappa = 500 / 200 = 2.5 \ll 20$: before the block size $b$ can exceed the correlation time ($b > 200$), fewer than 3 blocks remain, so the curve enters the high-variance finite-block noise regime before a plateau can form.
- [ ] **(B)** Because data blocking only works for Monte Carlo simulations, not Molecular Dynamics.
- [ ] **(C)** Because $\hat{\sigma}_{\bar{A}}(b)$ always decreases monotonically with $b$ when $\kappa > 100$.
- [ ] **(D)** Because $N = 500$ is not an exact power of 2.

In [26]:
ws.render_section('q_20')

**Q20 · medium** · 💡

According to Lecture 02 (Slide 15), we need at least **$N_{\text{eff}} = 20$ independent data points** to estimate statistical error bars reliably (so that the uncertainty in the error bar itself is $\sim 20\%$). If you are studying a phase transition where critical slowing down causes the statistical inefficiency to reach $\kappa = 350\text{ MC sweeps}$, what is the **minimum number of equilibrated MC sweeps** $N_{\text{eq}}$ you must run after warm-up?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Solve $N_{\text{eff}} = N_{\text{eq}} / \kappa = 20$ for $N_{\text{eq}}$.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('q_21')

**Q21 · hard** · 💡

For $M$ independent Gaussian block averages, the relative statistical error of the estimated standard error ('the error of the error') is given by $\frac{\Delta \hat{\sigma}}{\hat{\sigma}} \approx \frac{1}{\sqrt{2(M - 1)}}$. If you block your trajectory so that $M = 51$ independent blocks remain on the plateau, what is the percentage relative uncertainty $100 \times \frac{\Delta \hat{\sigma}}{\hat{\sigma}}$ in your reported error bar?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $M = 51$ into $100 / \sqrt{2(51 - 1)}$.

</div>
</details>

_Numeric answer required._

In [28]:
ws.render_section('sec_equilibration_warmup')

### The Equilibration (Warm-Up) Problem
When we start a simulation at step $k = 1$, our initial configuration $S_0$ is almost never drawn from the true canonical equilibrium distribution $F^*(S)$. For example, we might initialize a liquid simulation on a simple cubic crystal lattice, or start an Ising magnet with all spins pointing up ($\sigma_i = +1$).

As shown in the **Statistical Vocabulary** table (Lecture 02, Slide 16):
- The full raw **trace** is $A_k$ for $1 \le k \le N$.
- We must discard the initial non-equilibrium warm-up transient $1 \le k < k_1$ (often denoted $I_{\text{eq}}$) and retain only the **equilibrated data range** $k_1 \le k \le k_2$ (typically $k_2 = N$), which contains:
  $$N_{\text{eq}} = k_2 - k_1 + 1$$
  production steps.
- All equilibrium estimators — the mean $a = \frac{1}{N_{\text{eq}}}\sum_{k=k_1}^{k_2} A_k$, the sample variance $v = \frac{1}{N_{\text{eq}}-1}\sum_{k=k_1}^{k_2} (A_k - a)^2$, and the autocorrelation $c_A(i)$ — are evaluated **strictly over the equilibrated window $[k_1, k_2]$**.

### The Bias–Variance Trade-Off in Choosing $k_1$ ($I_{\text{eq}}$)
How should we choose the warm-up truncation step $k_1$?
- **If $k_1$ is too small ($k_1 \lesssim \tau_{\text{eq}}$):** The retained window still includes the exponential relaxation drift $\Delta A_0 e^{-k/\tau_{\text{eq}}}$ from the artificial initial state $S_0$. This contaminates the sample mean $a$ with a **systematic initialization bias** (and artificially inflates the sample variance $v$ and autocorrelation time $\kappa$!).
- **If $k_1$ is too large ($k_1 \to N$):** The remaining production window $N_{\text{eq}} = N - k_1 + 1$ becomes very short, reducing $N_{\text{eff}} = N_{\text{eq}}/\kappa$ and inflating the **statistical error** $\sigma_{\bar{A}} \propto N_{\text{eq}}^{-1/2}$.
- **Rule of thumb:** Choose $k_1$ to be at least **$5$ to $10$ times the relaxation/correlation time** ($k_1 \gtrsim 5\tau_{\text{eq}}$) where the trace visually and statistically flattens around a stationary mean, while ensuring $N_{\text{eq}} \gg 20\kappa$.

### Four Classic Slippery Traps in Simulation Analysis (Slide 17)
On Slide 17 of Lecture 02 (*"Statistical thinking is slippery: be careful"*), Prof. Trinkle highlights four misconceptions that trap almost every beginner:
1. **"Shouldn't the energy settle down to a constant in equilibrium?"**
   - **NO!** In any canonical ($NVT$) or grand-canonical ensemble (and for kinetic/potential energy separately in $NVE$), microscopic observables **fluctuate forever** with standard deviation $\sigma_A \propto 1/\sqrt{N_{\text{atoms}}}$. It is only the *overall sample mean* $\bar{A}$ that converges as $N_{\text{eq}} \to \infty$.
2. **"The cumulative energy curve $\bar{E}(n) = \frac{1}{n}\sum_{k=1}^n E_k$ looks smooth and flat at the end, so my simulation has converged!"**
   - **BEWARE!** Even pathological, non-ergodic, or still-drifting simulations produce deceptively smooth cumulative average curves at large $n$. Why? Because changing the $n$-th point by $\Delta E$ only changes the cumulative average $\bar{E}(n) = \frac{n-1}{n}\bar{E}(n-1) + \frac{1}{n}E_n$ by $\mathcal{O}(1/n)$, which acts as a heavy low-pass filter that hides late-time jumps or phase transitions! Always plot the **raw trace** and **windowed block averages**, never rely on a smooth cumulative mean alone.
3. **"Dataset A differs from Dataset B by 2 error bars, therefore the two models MUST be physically different!"**
   - **Not necessarily!** For a Gaussian distribution, a $\ge 2\sigma$ deviation occurs by pure chance about $5\%$ of the time ($\sim 1$ in $20$, or $\sim 1$ in $10$ when comparing pairs of noisy numbers). Conversely, *"if things agree too well (e.g. 20 parameters all within $0.1\sigma$), something is wrong!"*
4. **"My analysis procedure is too complicated to propagate errors analytically."**
   - **NO! NEVER!** Even for the most complex multi-step fitting workflow, you can **run your whole code $M = 10$ times** with independent random seeds (or split your equilibrated trajectory into $M = 10$ independent macro-blocks) and compute the mean and standard error directly from the variance across the $M$ independent outputs!


In [29]:
ws.render_section('concept_warmup')

### Interactive Explorer: Equilibration Cutoff k_1 (I_eq) & The Bias–Variance Trade-Off

Slide the warm-up truncation step $k_1$, relaxation timescale $\tau_{\text{eq}}$, and initial offset to see how discarding the initial transient eliminates **systematic warm-up bias** at the cost of a slightly shorter production window $N_{\text{eq}}$.

**Key Takeaway:** Setting $k_1 = 0$ includes the steep initial transient, creating a massive systematic bias in $\bar{A}$ and artificially inflating the apparent variance. Setting $k_1 \approx 3\tau_{\text{eq}}\text{ to }5\tau_{\text{eq}}$ reduces the transient bias below the statistical noise floor while preserving most of the trajectory for production averaging.

In [30]:
ws.render_section('q_22')

**Q22 · easy** · 💡

According to the Statistical Vocabulary table (Lecture 02, Slide 16), if a simulation generates a trace $A_k$ for $1 \le k \le 5{,}000$, and you discard the first $800$ steps as warm-up so that the equilibrated range begins at $k_1 = 801$ and ends at $k_2 = 5{,}000$, how many equilibrated data points $N_{\text{eq}} = k_2 - k_1 + 1$ are included in your production average?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $5{,}000 - 801 + 1$.

</div>
</details>

_Numeric answer required._

In [31]:
ws.render_section('q_23')

**Q23 · medium** · 💡

Suppose an MD potential energy trace of total length $N = 1{,}000$ steps has $E_k = -2.00\text{ eV}$ during the initial $200$-step warm-up ($1 \le k \le 200$) and fluctuates around a true equilibrium mean of $-3.00\text{ eV}$ during the remaining $800$ steps ($201 \le k \le 1{,}000$). If a student forgets to discard the warm-up period and averages over all $1{,}000$ steps, what is the resulting **systematic warm-up bias** $\bar{E}_{1..1000} - (-3.00\text{ eV})$ (in $\text{eV}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute the full-trace average $[200(-2.00) + 800(-3.00)] / 1000$, then subtract the true equilibrium mean $-3.00\text{ eV}$.

</div>
</details>

_Numeric answer required._

In [32]:
ws.render_section('q_24')

**Q24 · medium** · 💡

What happens to the **empirical sample variance** $v_A$ and **apparent autocorrelation time** $\kappa$ if you fail to discard a large initial equilibration transient before running your statistical analysis?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about what happens to $\sum (A_k - a)^2$ and low-frequency drift when $A_k$ starts far above the equilibrium mean and slowly decays.

</div>
</details>

- [ ] **(A)** Both the apparent variance $v_A$ and the apparent autocorrelation time $\kappa$ are artificially inflated by the slow, one-way transient drift.
- [ ] **(B)** The apparent variance $v_A$ becomes zero and $\kappa$ drops to $1$.
- [ ] **(C)** The sample mean $a$ is biased, but $v_A$ and $c_A(i)$ are mathematically unaffected by the transient.
- [ ] **(D)** The autocorrelation function $c_A(0)$ becomes negative.

In [33]:
ws.render_section('q_25')

**Q25 · medium** · 💡

Why does Slide 17 of Lecture 02 warn: *'"The cumulative energy has converged." — BEWARE. Even pathological cases have smooth cumulative energy curves'*?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Write the running cumulative mean as $\bar{E}_n = \frac{n-1}{n}\bar{E}_{n-1} + \frac{1}{n}E_n$ and consider the weight $1/n$ of new data at large $n$.

</div>
</details>

- [ ] **(A)** Because at large step count $n$, each new measurement $E_n$ is weighted by $1/n$, so the cumulative average $\bar{E}_n$ always looks visually smooth and sluggish even if the instantaneous trace $E_n$ just jumped to a new metastable state or is still drifting.
- [ ] **(B)** Because the cumulative energy is an integral that diverges to infinity for any negative potential energy.
- [ ] **(C)** Because cumulative averages can only be computed for uncorrelated random numbers.
- [ ] **(D)** Because floating-point addition always rounds the last digit to zero.

In [34]:
ws.render_section('q_26')

**Q26 · easy** · 💡

Suppose a student says: *'My simulation post-processing pipeline involves a complex non-linear curve fit and three coordinate transformations, so it is too complicated to compute error bars.'* What is Prof. Trinkle's direct prescription on Slide 17 of Lecture 02?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the fourth bullet on Slide 17: 'NO! NEVER! Run your whole code...'

</div>
</details>

- [ ] **(A)** Run the entire simulation/analysis pipeline $M = 10$ independent times (or on $M$ independent data blocks) and compute the mean and standard error directly from the variance of the $M$ results.
- [ ] **(B)** Omit the error bars and state in the caption that the algorithm is deterministic.
- [ ] **(C)** Report the machine epsilon ($10^{-16}$) as the error bar.
- [ ] **(D)** Replace the nonlinear observable with a linear one.

In [35]:
ws.render_section('q_27')

**Q27 · easy** · 💡

Following the prescription on Slide 17, you run a complex defect-migration simulation code $M = 10$ independent times with different random number seeds. Across the $10$ independent runs, the extracted migration barrier has a sample mean of $\bar{E}_m = 0.650\text{ eV}$ and a sample variance across the 10 runs of $s^2 = 0.00090\text{ eV}^2$ (so $s = 0.030\text{ eV}$). What is the **standard error of the mean** $\sigma_{\bar{E}_m} = s / \sqrt{M}$ across the 10 runs (in $\text{eV}$, to 4 decimal places)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $0.030 / \sqrt{10}$.

</div>
</details>

_Numeric answer required._

In [36]:
ws.render_section('sec_nonlinear_bias_and_resampling')

### Why $F(\bar{Z})$ Is Biased When $F$ Is Nonlinear (Lecture 02, Slide 13)
In statistical mechanics, many physical quantities are **nonlinear functions of ensemble expectations**, $F(\langle Z \rangle)$, rather than simple expectations of a single observable:
- **Binder Cumulant** (used to locate critical points in phase transitions):
  $$U_4 = 1 - \frac{\langle M^4 \rangle}{3\langle M^2 \rangle^2}$$
- **Arrhenius / Free-Energy Barriers:** $\Delta F = -k_B T \ln \langle e^{-\Delta H / k_B T} \rangle$
- **Ratios of expectations** (such as Virial compressibility or canonical reweighting $\frac{\langle A e^{-\beta \Delta H}\rangle}{\langle e^{-\beta \Delta H}\rangle}$).

Crucially, for any nonlinear function $F(Z)$, **the expectation of the function does NOT equal the function of the expectation**:
$$\langle F(Z) \rangle \ne F(\langle Z \rangle)$$
Suppose $Z = \bar{Z} + \delta Z$ is the sample mean from a Monte Carlo or MD run (with true mean $\bar{Z} = \langle Z \rangle$, zero mean fluctuation $\langle \delta Z \rangle = 0$, and standard error $\text{error}(Z) = \langle \delta Z^2 \rangle^{1/2} = \frac{\sigma_Z}{\sqrt{N_{\text{eff}}}}$), and we plug $Z$ into a nonlinear formula $F(Z)$. How large are the **systematic bias** and **statistical error** of $F(Z)$?

Following Slide 13 of Lecture 02, we expand $F(Z)$ in a Taylor power series around $\bar{Z}$:
$$F(Z) = F(\bar{Z}) + \left.\frac{dF}{dZ}\right|_{\bar{Z}} \delta Z + \frac{1}{2}\left.\frac{d^2 F}{dZ^2}\right|_{\bar{Z}} \delta Z^2 + \cdots$$

1. **Systematic Bias of $F(Z)$ — Scales as $\mathcal{O}(N^{-1})$:**
   Taking the expectation value of $F(Z) - F(\bar{Z})$ and using $\langle \delta Z \rangle = 0$ leaves the second-derivative curvature term:
   $$\boxed{\text{bias}(F) = \big\langle F(Z) - F(\bar{Z}) \big\rangle = \frac{1}{2}\left.\frac{d^2 F}{dZ^2}\right|_{\bar{Z}} \langle \delta Z^2 \rangle + \cdots \approx \frac{1}{2}\left.\frac{d^2 F}{dZ^2}\right|_{\bar{Z}} \text{error}(Z)^2 = \mathcal{O}(N_{\text{eff}}^{-1})}$$
   - If $F$ is strictly **linear** ($F''(\bar{Z}) = 0$), the bias is **identically zero**!
   - If $F$ is **convex** ($F''(\bar{Z}) > 0$, e.g. $F(Z) = Z^2$ or $F(Z) = 1/Z$ for $Z>0$), $\text{bias}(F) > 0$ (Jensen's inequality: your result is systematically *too high*).
   - If $F$ is **concave** ($F''(\bar{Z}) < 0$, e.g. $F(Z) = \ln Z$), $\text{bias}(F) < 0$ (your result is systematically *too low*).

2. **Statistical Error of $F(Z)$ — Scales as $\mathcal{O}(N^{-1/2})$:**
   Computing the root-mean-square fluctuation around the mean retains the leading first-derivative term:
   $$\boxed{\text{error}(F) = \big\langle (F(Z) - \langle F(Z) \rangle)^2 \big\rangle^{1/2} \approx \left|\left.\frac{dF}{dZ}\right|_{\bar{Z}}\right| \langle \delta Z^2 \rangle^{1/2} = \left|\left.\frac{dF}{dZ}\right|_{\bar{Z}}\right| \text{error}(Z) = \mathcal{O}(N_{\text{eff}}^{-1/2})}$$

Notice the asymptotic scaling hierarchy: as $N_{\text{eff}} \to \infty$, the systematic nonlinear bias shrinks as $\mathcal{O}(N_{\text{eff}}^{-1})$, which is faster than the statistical error $\mathcal{O}(N_{\text{eff}}^{-1/2})$. However, when $N_{\text{eff}}$ is modest or the curvature $|F''(\bar{Z})|$ is large, **you must estimate and correct for the bias**!

### Resampling Methods for Nonlinear Observables: Jackknife & Bootstrap
How do we estimate $\text{error}(F)$ and $\text{bias}(F)$ for an arbitrary nonlinear function $F(\bar{A}, \bar{B}, \dots)$ without taking analytical derivatives by hand? Suppose we first group our correlated trajectory into $M$ independent blocks ($b \gg \tau$) with block averages $B_1, B_2, \dots, B_M$, and let $\hat{\theta}_{\text{full}} = F(\bar{B})$ be the nonlinear estimator evaluated on the full dataset.

1. **Delete-1 Block Jackknife:**
   For each block $m \in \{1, \dots, M\}$, form the **$m$-th Jackknife subsample** by omitting block $m$ and averaging the remaining $M - 1$ blocks:
   $$\bar{B}_{(m)} = \frac{1}{M - 1}\sum_{j \ne m} B_j, \qquad \hat{\theta}_{(m)} = F\big(\bar{B}_{(m)}\big)$$
   Let $\bar{\theta}_{\text{JK}} = \frac{1}{M}\sum_{m=1}^M \hat{\theta}_{(m)}$ be the average of the $M$ Jackknife estimates. Because each $\hat{\theta}_{(m)}$ shares $M-2$ blocks with any other $\hat{\theta}_{(m')}$, their fluctuations are smaller by $1/(M-1)$, so the **Jackknife standard error** includes an $(M - 1)$ prefactor:
   $$\boxed{\sigma_{\text{JK}}(\hat{\theta}) = \sqrt{\frac{M - 1}{M}\sum_{m=1}^M \big(\hat{\theta}_{(m)} - \bar{\theta}_{\text{JK}}\big)^2}}$$
   Even better, since each Jackknife subsample has size $M - 1$ instead of $M$, its $\mathcal{O}(1/M)$ bias is larger by $\frac{M}{M-1}$, allowing us to extract the **Jackknife bias estimate** and the **bias-corrected estimator**:
   $$\widehat{\text{bias}}_{\text{JK}}(\hat{\theta}) = (M - 1)\big(\bar{\theta}_{\text{JK}} - \hat{\theta}_{\text{full}}\big), \qquad \hat{\theta}_{\text{corrected}} = \hat{\theta}_{\text{full}} - \widehat{\text{bias}}_{\text{JK}} = M\hat{\theta}_{\text{full}} - (M - 1)\bar{\theta}_{\text{JK}}$$

2. **Bootstrap Resampling:**
   Draw $M$ blocks *with replacement* from $\{B_1, \dots, B_M\}$ a large number of times $N_{\text{boot}}$ (e.g., $N_{\text{boot}} = 1{,}000$). For each bootstrap replicate $r$, compute the mean $\bar{B}^{*(r)}$ and evaluate $\hat{\theta}^{*(r)} = F(\bar{B}^{*(r)})$. The **Bootstrap standard error** is simply the sample standard deviation across the $N_{\text{boot}}$ replicates:
   $$\sigma_{\text{boot}}(\hat{\theta}) = \sqrt{\frac{1}{N_{\text{boot}} - 1}\sum_{r=1}^{N_{\text{boot}}} \big(\hat{\theta}^{*(r)} - \bar{\theta}^*\big)^2}, \qquad \widehat{\text{bias}}_{\text{boot}} = \bar{\theta}^* - \hat{\theta}_{\text{full}}$$


In [37]:
ws.render_section('concept_nonlinear_bias')

### Interactive Explorer: Nonlinear Estimator Bias O(N⁻¹) vs. Statistical Error O(N⁻¹/²)

Adjust the second-derivative curvature $F''(\bar{Z})$ and fluctuation width $\sigma_Z$ on a log–log plot to compare the $\mathcal{O}(N_{\text{eff}}^{-1/2})$ statistical error against the $\mathcal{O}(N_{\text{eff}}^{-1})$ systematic nonlinear bias (Lecture 02, Slide 13).

**Key Takeaway:** On a log–log plot against $N_{\text{eff}}$, statistical error has slope $-1/2$ while nonlinear bias has a steeper slope of $-1$. At large $N_{\text{eff}}$ statistical error dominates, but at small $N_{\text{eff}}$ or large curvature $F''(\bar{Z})$, nonlinear bias can be comparable to the error bar!

In [38]:
ws.render_section('q_28')

**Q28 · easy** · 💡

When a random sample average $Z = \bar{Z} + \delta Z$ with standard error $\text{error}(Z) \propto N^{-1/2}$ is plugged into a smooth nonlinear function $F(Z)$ (with $F'(\bar{Z}) \ne 0$ and $F''(\bar{Z}) \ne 0$), how do the **systematic bias** $\text{bias}(F) = \langle F(Z) - F(\bar{Z}) \rangle$ and the **statistical error** $\text{error}(F)$ scale with the number of independent samples $N$ (Lecture 02, Slide 13)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the red $\mathcal{O}(\cdot)$ annotations on Slide 13 of Lecture 02.

</div>
</details>

- [ ] **(A)** $\text{bias}(F)$ scales as $\mathcal{O}(N^{-1})$, while $\text{error}(F)$ scales as $\mathcal{O}(N^{-1/2})$.
- [ ] **(B)** Both $\text{bias}(F)$ and $\text{error}(F)$ scale as $\mathcal{O}(N^{-1/2})$.
- [ ] **(C)** $\text{bias}(F)$ is independent of $N$ ($\mathcal{O}(1)$), while $\text{error}(F)$ scales as $\mathcal{O}(N^{-1})$.
- [ ] **(D)** $\text{bias}(F)$ scales as $\mathcal{O}(N^{-1/2})$, while $\text{error}(F)$ scales as $\mathcal{O}(N^{-1})$.

In [39]:
ws.render_section('q_29')

**Q29 · medium** · 💡

Suppose you estimate a variable $Z$ from a Monte Carlo simulation with mean $\bar{Z} = 2.0$ and standard error $\text{error}(Z) = \langle \delta Z^2 \rangle^{1/2} = 0.10$. You want to report the nonlinear quantity $F(Z) = Z^2$. Using the Taylor formula on Slide 13:
$$\text{bias}(F) = \langle F(Z) - F(\bar{Z}) \rangle = \frac{1}{2}\left.\frac{d^2 F}{dZ^2}\right|_{\bar{Z}} \text{error}(Z)^2$$
what is the exact systematic bias $\text{bias}(F)$ of $Z^2$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

For $F(Z) = Z^2$, the second derivative is $d^2F/dZ^2 = 2$. Compute $\frac{1}{2}(2)(0.10)^2$.

</div>
</details>

_Numeric answer required._

In [40]:
ws.render_section('q_30')

**Q30 · medium** · 💡

For the same variable $Z$ with mean $\bar{Z} = 2.0$ and standard error $\text{error}(Z) = 0.10$, what is the leading-order **statistical error** $\text{error}(F) \approx \left|\left.\frac{dF}{dZ}\right|_{\bar{Z}}\right| \text{error}(Z)$ of $F(Z) = Z^2$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $|dF/dZ| = 2\bar{Z} = 4.0$ at $\bar{Z} = 2.0$, and multiply by $\text{error}(Z) = 0.10$.

</div>
</details>

_Numeric answer required._

In [41]:
ws.render_section('q_31')

**Q31 · hard** · 💡

In a free-energy perturbation calculation, you measure an exponential Boltzmann average $Z$ with true mean $\bar{Z} = 1.0$ and standard error $\text{error}(Z) = 0.20$, and then compute the free energy via the nonlinear logarithm $F(Z) = -\ln(Z)$ (in units of $k_B T = 1$). Using $\text{bias}(F) \approx \frac{1}{2}F''(\bar{Z})\,\text{error}(Z)^2$, what is the leading-order systematic bias $\text{bias}(F)$ (including the correct sign)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

For $F(Z) = -\ln(Z)$, we have $F'(Z) = -1/Z$ and $F''(Z) = +1/Z^2$. Evaluate $\frac{1}{2}F''(1.0)(0.20)^2$.

</div>
</details>

_Numeric answer required._

In [42]:
ws.render_section('q_32')

**Q32 · hard** · 💡

You partition an equilibrated trajectory into $M = 5$ independent blocks to perform a **Delete-1 Block Jackknife** analysis on the Binder cumulant $U_4$. Leaving out one block at a time gives the five Jackknife estimates:
$$\hat{\theta}_{(1)} = 0.60,\quad \hat{\theta}_{(2)} = 0.62,\quad \hat{\theta}_{(3)} = 0.61,\quad \hat{\theta}_{(4)} = 0.58,\quad \hat{\theta}_{(5)} = 0.59$$
whose mean is $\bar{\theta}_{\text{JK}} = 0.60$. Using the Jackknife standard error formula:
$$\sigma_{\text{JK}} = \sqrt{\frac{M - 1}{M}\sum_{m=1}^M \big(\hat{\theta}_{(m)} - \bar{\theta}_{\text{JK}}\big)^2}$$
what is $\sigma_{\text{JK}}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First compute $\sum_{m=1}^5 (\hat{\theta}_{(m)} - 0.60)^2 = 0^2 + 0.02^2 + 0.01^2 + (-0.02)^2 + (-0.01)^2 = 0.0010$. Multiply by $(M-1)/M = 4/5$, then take the square root.

</div>
</details>

_Numeric answer required._

In [43]:
ws.render_section('q_33')

**Q33 · hard** · 💡

In a $M = 10$-block Jackknife analysis of a nonlinear susceptibility ratio, the full-sample estimate is $\hat{\theta}_{\text{full}} = 2.500$, while the mean of the $M = 10$ delete-one-block Jackknife estimates is $\bar{\theta}_{\text{JK}} = 2.502$. What is the **bias-corrected Jackknife estimate** $\hat{\theta}_{\text{corrected}} = \hat{\theta}_{\text{full}} - (M - 1)(\bar{\theta}_{\text{JK}} - \hat{\theta}_{\text{full}})$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\widehat{\text{bias}}_{\text{JK}} = 9 \times (2.502 - 2.500) = +0.018$, and subtract it from $\hat{\theta}_{\text{full}} = 2.500$.

</div>
</details>

_Numeric answer required._

In [44]:
ws.render_section('q_34')

**Q34 · medium** · 💡

Why does the Delete-1 Jackknife variance formula $\sigma_{\text{JK}}^2 = \frac{M-1}{M}\sum_{m=1}^M (\hat{\theta}_{(m)} - \bar{\theta}_{\text{JK}})^2$ have the prefactor $\frac{M-1}{M}$ in the numerator instead of the usual $\frac{1}{M(M-1)}$ used for $M$ independent block averages?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How much data do $\hat{\theta}_{(1)}$ and $\hat{\theta}_{(2)}$ share in common?

</div>
</details>

- [ ] **(A)** Because each delete-1 Jackknife subsample $\hat{\theta}_{(m)}$ averages $M-1$ of the $M$ blocks, so two Jackknife estimates overlap on $M-2$ blocks and vary among themselves by a factor of $1/(M-1)$ less than individual block averages.
- [ ] **(B)** Because Jackknife estimates have infinite variance unless multiplied by $M-1$.
- [ ] **(C)** Because the Jackknife method assumes the data follow a Cauchy distribution.
- [ ] **(D)** Because only $1$ block is kept in each Jackknife subsample and $M-1$ blocks are discarded.

In [45]:
ws.render_section('sec_end_to_end_protocol')

## 6. Putting It All Together: The Complete MSE 485 Error-Analysis Protocol

Let's synthesize the complete, step-by-step error analysis protocol from Lecture 02 (Slide 16: **Statistical Vocabulary**) that you will use on every homework assignment and project in MSE 485:

```
Raw Simulation Trace A_k (1 <= k <= N)
   │
   ▼
[Step 1: Equilibration Truncation]
   Plot trace A_k vs. k; identify initial transient decay.
   Discard warm-up steps k < k_1; keep equilibrated range k_1 <= k <= k_2 (N_eq = k_2 - k_1 + 1).
   │
   ▼
[Step 2: Sample Mean & Variance]
   Compute mean:     a = (1 / N_eq) * sum_{k=k_1}^{k_2} A_k
   Compute variance: v = [1 / (N_eq - 1)] * sum_{k=k_1}^{k_2} (A_k - a)^2,   sigma = sqrt(v)
   │
   ├──► [Method A: Autocorrelation Function]
   │       c_A(i) = (1 / v) * [1 / (N_eq - i)] * sum_{k=k_1}^{k_2 - i} (A_k - a)(A_{k+i} - a)
   │       Find first zero crossing i_cutoff where c_A(i) <= 0.
   │       Compute statistical inefficiency:  kappa = 1 + 2 * sum_{i=1}^{i_cutoff} c_A(i)
   │
   └──► [Method B: Flyvbjerg-Petersen Data Blocking]
           Group N_eq points into M = N_eq / b blocks of size b = 1, 2, 4, 8, ...
           Plot block error sigma_hat(b) = sqrt(sigma^2(B) / M) vs. b.
           Identify flat plateau at b >> tau (while M >= 20); extract kappa = (sigma_plateau / sigma_naive)^2.
   │
   ▼
[Step 3: Effective Sample Size & True Standard Error]
   Compute N_eff = N_eq / kappa   (Verify N_eff >= 20!)
   Report:  <A> = a +/- sqrt(v / N_eff) = a +/- sqrt(v * kappa / N_eq)
   │
   ▼
[Step 4: Nonlinear Functions F(<A>, <B>)]
   Apply Delete-1 Block Jackknife or Bootstrap over M >= 20 independent blocks (b >> tau)
   to obtain bias-corrected F and its standard error sigma_F.
```


In [46]:
ws.render_section('q_35')

**Q35 · easy** · 💡

In the **Statistical Vocabulary** table (Lecture 02, Slide 16), which formula defines the unbiased finite-sample estimator $c_A(i)$ for the autocorrelation function at lag $i$ over the equilibrated range $k_1 \le k \le k_2$ ($N_{\text{eq}} = k_2 - k_1 + 1$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice that for lag $i$, there are $N_{\text{eq}} - i$ valid pairs $(A_k, A_{k+i})$ inside $[k_1, k_2]$, normalized by the sample variance $v$.

</div>
</details>

- [ ] **(A)** $c_A(i) = \dfrac{1}{v}\dfrac{1}{N_{\text{eq}} - i}\displaystyle\sum_{k=k_1}^{k_2 - i} (A_k - a)(A_{k+i} - a)$
- [ ] **(B)** $c_A(i) = \dfrac{1}{N_{\text{eq}}}\displaystyle\sum_{k=1}^{N} A_k A_{k+i}$
- [ ] **(C)** $c_A(i) = \dfrac{v}{N_{\text{eq}}}\displaystyle\sum_{k=k_1}^{k_2 - i} (A_k + a)(A_{k+i} + a)$
- [ ] **(D)** $c_A(i) = 1 + 2\displaystyle\sum_{k=1}^{i} A_k$

In [47]:
ws.render_section('q_36')

**Q36 · medium** · 💡

Using the estimator formula $c_A(i) = \frac{1}{v(N_{\text{eq}} - i)}\sum_{k=1}^{N_{\text{eq}}-i}(A_k - a)(A_{k+i} - a)$, consider a tiny toy trace of $N_{\text{eq}} = 4$ equilibrated points: $A = [2.0,\ 4.0,\ 2.0,\ 0.0]$, which has sample mean $a = 2.0$ and sample variance $v = \frac{(0)^2 + (2)^2 + (0)^2 + (-2)^2}{4 - 1} = \frac{8}{3}$. What is the value of $c_A(1)$ at lag $i = 1$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\delta A = A - a = [0,\ +2,\ 0,\ -2]$. Sum the $N_{\text{eq}} - 1 = 3$ products $\delta A_1 \delta A_2 + \delta A_2 \delta A_3 + \delta A_3 \delta A_4$, then divide by $3 v$.

</div>
</details>

_Numeric answer required._

In [48]:
ws.render_section('q_37')

**Q37 · medium** · 💡

For the same 4-point toy trace $A = [2.0,\ 4.0,\ 2.0,\ 0.0]$ with $\delta A = [0,\ +2,\ 0,\ -2]$ and sample variance $v = 8/3$, what is the value of $c_A(2)$ at lag $i = 2$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

At lag $i = 2$, there are $N_{\text{eq}} - 2 = 2$ pairs: $\delta A_1 \delta A_3 + \delta A_2 \delta A_4 = (0)(0) + (2)(-2) = -4$. Divide by $(N_{\text{eq}} - 2)v = 2 \times (8/3) = 16/3$.

</div>
</details>

_Numeric answer required._

In [49]:
ws.render_section('q_38')

**Q38 · hard** · 💡

You perform a complete Molecular Dynamics simulation of $N = 25{,}000$ total time steps. Visual and statistical inspection shows that equilibration is complete after $5{,}000$ steps, so you analyze the equilibrated range $k_1 = 5{,}001$ to $k_2 = 25{,}000$ ($N_{\text{eq}} = 20{,}000$). In this production window, the total energy has sample variance $v_A = 0.360\text{ eV}^2$, and the autocorrelation sum up to the first zero crossing is $\sum_{i=1}^{i_{\text{cutoff}}} c_A(i) = 9.50$. What is the **true standard error of the mean** $\sigma_{\bar{A}}$ (in $\text{eV}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First compute $\kappa = 1 + 2(9.50) = 20.0$. Then find $N_{\text{eff}} = N_{\text{eq}} / \kappa = 20{,}000 / 20.0 = 1{,}000$, and evaluate $\sqrt{v_A / N_{\text{eff}}}$.

</div>
</details>

_Numeric answer required._

In [50]:
ws.render_section('q_39')

**Q39 · medium** · 💡

When applying **Bootstrap** or **Jackknife** resampling to a time series from a Molecular Dynamics or Monte Carlo simulation with correlation time $\kappa = 50$ steps, why is it a **grave error** to resample individual time steps $A_k$ independently instead of resampling **blocks** of length $b \gg \kappa$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What happens to the sequential time correlations if you scramble individual steps $A_k$ randomly with replacement?

</div>
</details>

- [ ] **(A)** Resampling individual steps destroys the sequential time correlation structure and treats all $N_{\text{eq}}$ steps as independent ($\kappa = 1$), underestimating the true standard error by a factor of $\sqrt{\kappa} = \sqrt{50} \approx 7.1\times$.
- [ ] **(B)** Resampling individual steps changes the sample mean to zero.
- [ ] **(C)** Bootstrap resampling can only be applied to datasets with fewer than 10 points.
- [ ] **(D)** Resampling individual steps overestimates the error bar by a factor of $\kappa^2$.

In [51]:
ws.render_section('q_40')

**Q40 · medium** · 💡

In a canonical ($NVT$) simulation of a magnetic crystal, the magnetic susceptibility is related to the variance of the magnetization $M$ by the fluctuation–dissipation formula $\chi = \frac{1}{k_B T}\big(\langle M^2 \rangle - \langle M \rangle^2\big)$. Suppose $k_B T = 2.0$, and across $M_{\text{blocks}} = 25$ independent blocks you estimate $\bar{M} = 0.0$ with standard error $\text{error}(\bar{M}) = 0.20$, while $\overline{M^2} = 1.00$ has negligible error. Because $\hat{\chi} = \frac{1}{k_B T}\big(\overline{M^2} - \bar{M}^2\big)$ contains the nonlinear term $-\frac{1}{k_B T}\bar{M}^2$, what is the **systematic bias** $\text{bias}(\hat{\chi}) = \langle \hat{\chi} - \chi_{\text{true}} \rangle$ caused by squaring the noisy sample mean $\bar{M}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Here $F(\bar{M}) = -\frac{1}{2 k_B T}\cdot 2\bar{M}^2 = -\frac{1}{2.0}\bar{M}^2 = -0.5\,\bar{M}^2$, so $F''(0) = -1.0$. Use $\text{bias}(F) = \frac{1}{2}F''(0)\,\text{error}(\bar{M})^2$.

</div>
</details>

_Numeric answer required._

In [52]:
ws.render_section('q_41')

**Q41 · medium** · 💡

Select **all** statements from Lecture 02 that represent sound, rigorous practice when analyzing and reporting atomistic simulation data:

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review Slides 4, 15, 16, and 17 of Lecture 02.

</div>
</details>

- [ ] **(A)** Every reported simulation quantity must include an error bar; results without error estimates are only suggestive.
- [ ] **(B)** Both the sample mean and its statistical error bar can be simultaneously determined from the same equilibrated trajectory once time correlations $\kappa$ are accounted for.
- [ ] **(C)** To ensure the error of the error estimate is at most $\sim 20\%$, the equilibrated trajectory must span at least $N_{\text{eff}} = N_{\text{eq}} / \kappa \ge 20$ effective independent samples.
- [ ] **(D)** Running a simulation longer ($N_{\text{eq}} \to \infty$) automatically eliminates systematic errors such as interatomic potential inaccuracies and finite-timestep drift.

In [53]:
ws.render_section('q_42')

**Q42 · hard** · 💡

Suppose you are comparing two simulation runs of the same liquid at the same $(N, V, T)$: Run 1 saves configurations every $\Delta t_1 = 1\text{ fs}$ for $100\text{ ps}$ ($N_1 = 100{,}000$ points, with correlation time $\kappa_1 = 200$ saved steps), while Run 2 saves configurations every $\Delta t_2 = 100\text{ fs}$ for the same $100\text{ ps}$ physical duration ($N_2 = 1{,}000$ points, with correlation time $\kappa_2 \approx 2$ saved steps). Assuming $100\text{ fs}$ is smaller than the physical correlation time ($200\text{ fs}$), which statements are true? Select **all** that apply.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare $N_{\text{eff},1} = N_1 / \kappa_1$ and $N_{\text{eff},2} = N_2 / \kappa_2$ over the same $100\text{ ps}$ physical trajectory!

</div>
</details>

- [ ] **(A)** Both runs span the exact same physical duration ($100\text{ ps}$) and contain essentially the same number of effective independent samples ($N_{\text{eff}} \approx 500$), yielding the same true standard error of the mean.
- [ ] **(B)** If someone mistakenly applies the naive uncorrelated formula $s_A / \sqrt{N}$, Run 1 will falsely appear to have a $10\times$ smaller error bar than Run 2 simply because it wrote to disk $100\times$ more frequently.
- [ ] **(C)** In a Flyvbjerg–Petersen data blocking plot against physical block duration $t_b = b \Delta t$, both runs converge to the same plateau error bar once $t_b \gg 200\text{ fs}$.
- [ ] **(D)** Saving configurations $100\times$ more frequently during a fixed $100\text{ ps}$ MD trajectory reduces the true statistical error bar by a factor of $10$.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()